# FastAPI ile Kütüphane Yönetim API'si (Production-Grade Örnek)

**İş Problemi:** [22.6.1-Python_ile_API_Geliştirmek_FastAPI.pdf](22.6.1-Python_ile_API_Geliştirmek_FastAPI.pdf)
dosyasında anlattığımız katmanlı mimariyi (veri, şema, kimlik doğrulama, iş mantığı, sunum) uçtan
uca, gerçek bir veritabanına (SQLite + SQLAlchemy) yazan, JWT ile korunan, sayfalanan ve
filtrelenen, tam CRUD destekli bir REST API'ye dönüştürüyoruz.

Bu notebook'u çalıştırmak API'yi bir ağ portunda AYAĞA KALDIRMAZ; bunun yerine FastAPI'nin resmi
test aracı olan `TestClient` kullanılır (bkz. 22.6.1, Bölüm 3.1) - bu sayede notebook tamamen
offline, hızlı ve tekrar üretilebilir şekilde çalışır. En altta, gerçek bir sunucu olarak nasıl
çalıştırılacağı da (uvicorn ile) ayrıca gösterilmiştir.

Gerekli kütüphaneleri kurmak için: `pip install fastapi uvicorn sqlalchemy pyjwt python-multipart pytest`

## 1. Veri Katmanı (SQLAlchemy ORM Modelleri)

22.6.1, Bölüm 3'te anlatıldığı gibi: kitaplar ve kullanıcılar gerçek bir SQLite veritabanı
tablosunda saklanır (bkz. 15-SQL modülü, ilişkisel veritabanı kavramları).

In [1]:
import hashlib
import hmac
import os
import time
from datetime import datetime, timedelta, timezone
from typing import Optional, List

import jwt
from fastapi import FastAPI, Depends, HTTPException, status, Query
from fastapi.middleware.cors import CORSMiddleware
from fastapi.security import OAuth2PasswordBearer, OAuth2PasswordRequestForm
from fastapi.testclient import TestClient
from pydantic import BaseModel, Field, ConfigDict
from sqlalchemy import create_engine, Column, Integer, String, Boolean, ForeignKey
from sqlalchemy.orm import declarative_base, relationship, sessionmaker, Session
from sqlalchemy.pool import StaticPool

Base = declarative_base()


class KullaniciDB(Base):
    __tablename__ = "kullanicilar"
    id = Column(Integer, primary_key=True, index=True)
    kullanici_adi = Column(String, unique=True, index=True, nullable=False)
    sifre_hash = Column(String, nullable=False)
    kitaplar = relationship("KitapDB", back_populates="sahip")


class KitapDB(Base):
    __tablename__ = "kitaplar"
    id = Column(Integer, primary_key=True, index=True)
    baslik = Column(String, index=True, nullable=False)
    yazar = Column(String, index=True, nullable=False)
    yil = Column(Integer, nullable=True)
    musait_mi = Column(Boolean, default=True)
    sahip_id = Column(Integer, ForeignKey("kullanicilar.id"))
    sahip = relationship("KullaniciDB", back_populates="kitaplar")

print("ORM modelleri tanımlandı.")

C:\Users\ONI\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\fastapi\testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa


ORM modelleri tanımlandı.


## 2. Şema Katmanı (Pydantic Modelleri)

22.6.1, Bölüm 3'te vurgulandığı gibi: veritabanı modelleri (yukarıda) ile API'nin dışarıya
gösterdiği şemalar (aşağıda) BİLİNÇLİ olarak AYRILIR — örn. `sifre_hash` ASLA dışarı sızmaz.

In [2]:
class KullaniciOlustur(BaseModel):
    kullanici_adi: str = Field(..., min_length=3, max_length=50, examples=["ayse_yilmaz"])
    sifre: str = Field(..., min_length=6, examples=["guclu-sifre-123"])


class KullaniciYanit(BaseModel):
    model_config = ConfigDict(from_attributes=True)
    id: int
    kullanici_adi: str


class KitapOlustur(BaseModel):
    baslik: str = Field(..., min_length=1, max_length=200, examples=["Suç ve Ceza"])
    yazar: str = Field(..., min_length=1, max_length=100, examples=["Dostoyevski"])
    yil: Optional[int] = Field(None, ge=0, le=2100, examples=[1866])


class KitapGuncelle(BaseModel):
    """PATCH için - tüm alanlar opsiyonel, sadece gönderilen alan değişir (bkz. 22.2.1, Bölüm 3)."""
    baslik: Optional[str] = None
    yazar: Optional[str] = None
    yil: Optional[int] = None
    musait_mi: Optional[bool] = None


class KitapYanit(BaseModel):
    model_config = ConfigDict(from_attributes=True)
    id: int
    baslik: str
    yazar: str
    yil: Optional[int]
    musait_mi: bool
    sahip_id: int


class KitapSayfaYaniti(BaseModel):
    """22.3.1, Bölüm 4'te anlatılan offset tabanlı sayfalama yanıt zarfı (envelope)."""
    toplam: int
    sayfa: int
    limit: int
    kitaplar: List[KitapYanit]


class TokenYaniti(BaseModel):
    access_token: str
    token_type: str = "bearer"

print("Pydantic şemaları tanımlandı.")

Pydantic şemaları tanımlandı.


## 3. Güvenlik Yardımcıları (Şifre Hash'leme + JWT)

Eğitim ortamında kurulum kolaylığı için (bcrypt bazı ortamlarda derleme sorunu çıkarabiliyor),
Python'un yerleşik `hashlib.pbkdf2_hmac` fonksiyonuyla, tuzlanmış (salted) şifre hash'leme
uyguluyoruz. Production'da `passlib[bcrypt]`/`argon2` gibi özel tasarlanmış kütüphaneler tercih
edilmelidir; temel prensip (asla düz metin şifre saklamamak) aynıdır.

In [3]:
GIZLI_ANAHTAR = os.environ.get("KUTUPHANE_API_GIZLI_ANAHTAR", "egitim-amacli-demo-anahtari-asla-production-da-kullanma")
JWT_ALGORITMA = "HS256"
TOKEN_GECERLILIK_DAKIKA = 30


def sifreyi_hashle(sifre: str) -> str:
    tuz = os.urandom(16)
    hash_deger = hashlib.pbkdf2_hmac("sha256", sifre.encode(), tuz, 100_000)
    return tuz.hex() + ":" + hash_deger.hex()


def sifre_dogru_mu(sifre: str, kayitli_hash: str) -> bool:
    tuz_hex, hash_hex = kayitli_hash.split(":")
    tuz = bytes.fromhex(tuz_hex)
    beklenen = hashlib.pbkdf2_hmac("sha256", sifre.encode(), tuz, 100_000)
    return hmac.compare_digest(beklenen.hex(), hash_hex)


def jwt_token_uret(kullanici_adi: str) -> str:
    """bkz. 22.4.1, Bölüm 2.3 - JWT'nin 3 parçası (header, payload, signature)."""
    son_kullanma = datetime.now(timezone.utc) + timedelta(minutes=TOKEN_GECERLILIK_DAKIKA)
    payload = {"sub": kullanici_adi, "exp": son_kullanma}
    return jwt.encode(payload, GIZLI_ANAHTAR, algorithm=JWT_ALGORITMA)

print("Güvenlik yardımcıları hazır.")

Güvenlik yardımcıları hazır.


## 4. Veritabanı Bağlantısı ve Bağımlılık (Dependency) Fonksiyonları

Bellek-içi (in-memory) SQLite + `StaticPool`: TÜM bağlantılar AYNI tek veritabanı bağlantısını
paylaşır, böylece veri kaybolmaz (normal `:memory:` her yeni bağlantıda BOŞ bir DB verir) ama
hiçbir DOSYA da diske yazılmaz. Bu, notebook her çalıştırıldığında SIFIRDAN, tamamen izole bir
veritabanıyla başlamasını garanti eder — test kirliliği (test pollution) riskini ortadan kaldırır.

In [4]:
DATABASE_URL = "sqlite:///:memory:"
engine = create_engine(DATABASE_URL, connect_args={"check_same_thread": False}, poolclass=StaticPool)
SessionLocal = sessionmaker(autocommit=False, autoflush=False, bind=engine)
Base.metadata.create_all(bind=engine)

oauth2_scheme = OAuth2PasswordBearer(tokenUrl="token")


def get_db():
    """22.6.1, Bölüm 2.3'te anlatılan Dependency Injection örneği: her istek için AYRI bir
    veritabanı oturumu açılır ve istek bitince otomatik kapatılır."""
    db = SessionLocal()
    try:
        yield db
    finally:
        db.close()


def mevcut_kullaniciyi_getir(token: str = Depends(oauth2_scheme), db: Session = Depends(get_db)) -> KullaniciDB:
    """Bu da bir Dependency: Authorization: Bearer <token> başlığını çözüp, geçerli kullanıcıyı
    döndürür. Kimlik doğrulama gerektiren HER endpoint sadece bu fonksiyonu Depends() ile çağırır."""
    kimlik_hatasi = HTTPException(
        status_code=status.HTTP_401_UNAUTHORIZED,
        detail="Kimlik doğrulanamadı",
        headers={"WWW-Authenticate": "Bearer"},
    )
    try:
        payload = jwt.decode(token, GIZLI_ANAHTAR, algorithms=[JWT_ALGORITMA])
        kullanici_adi = payload.get("sub")
        if kullanici_adi is None:
            raise kimlik_hatasi
    except jwt.PyJWTError:
        raise kimlik_hatasi

    kullanici = db.query(KullaniciDB).filter(KullaniciDB.kullanici_adi == kullanici_adi).first()
    if kullanici is None:
        raise kimlik_hatasi
    return kullanici

print("Veritabanı ve dependency fonksiyonları hazır.")

Veritabanı ve dependency fonksiyonları hazır.


## 5. FastAPI Uygulaması, CORS Ayarları ve Uç Noktalar (Path Operations)

bkz. 22.4.1, Bölüm 4.2 ve 22.6.1, Bölüm 4 — production'da ASLA `allow_origins=["*"]` kullanılmaz.

In [5]:
app = FastAPI(
    title="Kütüphane Yönetim API'si",
    description="22-API modülü, 22.6 bölümü için örnek üretim-kalitesinde REST API.",
    version="1.0.0",
)

app.add_middleware(
    CORSMiddleware,
    allow_origins=["http://localhost:3000"],
    allow_methods=["*"],
    allow_headers=["*"],
)


@app.post("/register", response_model=KullaniciYanit, status_code=status.HTTP_201_CREATED, tags=["Kimlik Doğrulama"])
def kayit_ol(veri: KullaniciOlustur, db: Session = Depends(get_db)):
    """Yeni kullanıcı oluşturur. Kullanıcı adı benzersiz olmalıdır."""
    mevcut = db.query(KullaniciDB).filter(KullaniciDB.kullanici_adi == veri.kullanici_adi).first()
    if mevcut:
        raise HTTPException(status_code=status.HTTP_400_BAD_REQUEST, detail="Bu kullanıcı adı zaten kayıtlı")
    yeni_kullanici = KullaniciDB(kullanici_adi=veri.kullanici_adi, sifre_hash=sifreyi_hashle(veri.sifre))
    db.add(yeni_kullanici)
    db.commit()
    db.refresh(yeni_kullanici)
    return yeni_kullanici


@app.post("/token", response_model=TokenYaniti, tags=["Kimlik Doğrulama"])
def giris_yap(form_data: OAuth2PasswordRequestForm = Depends(), db: Session = Depends(get_db)):
    """bkz. 22.4.1, Bölüm 2.3 - kullanıcı adı/şifre doğrulanır, karşılığında kısa ömürlü bir
    JWT access token döner."""
    kullanici = db.query(KullaniciDB).filter(KullaniciDB.kullanici_adi == form_data.username).first()
    if not kullanici or not sifre_dogru_mu(form_data.password, kullanici.sifre_hash):
        raise HTTPException(
            status_code=status.HTTP_401_UNAUTHORIZED,
            detail="Kullanıcı adı veya şifre hatalı",
            headers={"WWW-Authenticate": "Bearer"},
        )
    return {"access_token": jwt_token_uret(kullanici.kullanici_adi), "token_type": "bearer"}


@app.get("/me", response_model=KullaniciYanit, tags=["Kimlik Doğrulama"])
def profilim(mevcut_kullanici: KullaniciDB = Depends(mevcut_kullaniciyi_getir)):
    return mevcut_kullanici


@app.post("/books", response_model=KitapYanit, status_code=status.HTTP_201_CREATED, tags=["Kitaplar"])
def kitap_ekle(
    veri: KitapOlustur,
    mevcut_kullanici: KullaniciDB = Depends(mevcut_kullaniciyi_getir),
    db: Session = Depends(get_db),
):
    """Yeni kitap ekler. Kimlik doğrulama ZORUNLUDUR (bkz. 22.4.1)."""
    yeni_kitap = KitapDB(**veri.model_dump(), sahip_id=mevcut_kullanici.id)
    db.add(yeni_kitap)
    db.commit()
    db.refresh(yeni_kitap)
    return yeni_kitap


@app.get("/books", response_model=KitapSayfaYaniti, tags=["Kitaplar"])
def kitaplari_listele(
    sayfa: int = Query(1, ge=1, description="Sayfa numarası (1'den başlar)"),
    limit: int = Query(10, ge=1, le=100, description="Sayfa başına kayıt sayısı"),
    yazar: Optional[str] = Query(None, description="Yazara göre filtrele (kısmi eşleşme)"),
    sadece_musait: bool = Query(False, description="Sadece ödünç alınabilir kitapları göster"),
    db: Session = Depends(get_db),
):
    """Sayfalanmış, filtrelenebilir kitap listesi. KİMLİK DOĞRULAMA GEREKTİRMEZ."""
    sorgu = db.query(KitapDB)
    if yazar:
        sorgu = sorgu.filter(KitapDB.yazar.ilike(f"%{yazar}%"))
    if sadece_musait:
        sorgu = sorgu.filter(KitapDB.musait_mi == True)  # noqa: E712

    toplam = sorgu.count()
    kitaplar = sorgu.offset((sayfa - 1) * limit).limit(limit).all()
    return {"toplam": toplam, "sayfa": sayfa, "limit": limit, "kitaplar": kitaplar}


@app.get("/books/{kitap_id}", response_model=KitapYanit, tags=["Kitaplar"])
def kitap_getir(kitap_id: int, db: Session = Depends(get_db)):
    kitap = db.query(KitapDB).filter(KitapDB.id == kitap_id).first()
    if not kitap:
        raise HTTPException(status_code=status.HTTP_404_NOT_FOUND, detail=f"{kitap_id} numaralı kitap bulunamadı")
    return kitap


def _kitabi_ve_sahiplik_kontrolunu_yap(kitap_id: int, mevcut_kullanici: KullaniciDB, db: Session) -> KitapDB:
    """22.4.1, Bölüm 4'teki BOLA (Broken Object Level Authorization) riskine karşı örnek önlem:
    sadece token'ın geçerli olması yetmez, kaynağın GERÇEKTEN bu kullanıcıya ait olup olmadığı da
    KONTROL EDİLİR."""
    kitap = db.query(KitapDB).filter(KitapDB.id == kitap_id).first()
    if not kitap:
        raise HTTPException(status_code=status.HTTP_404_NOT_FOUND, detail=f"{kitap_id} numaralı kitap bulunamadı")
    if kitap.sahip_id != mevcut_kullanici.id:
        raise HTTPException(status_code=status.HTTP_403_FORBIDDEN, detail="Bu kitabı değiştirme yetkiniz yok")
    return kitap


@app.patch("/books/{kitap_id}", response_model=KitapYanit, tags=["Kitaplar"])
def kitap_kismi_guncelle(
    kitap_id: int,
    veri: KitapGuncelle,
    mevcut_kullanici: KullaniciDB = Depends(mevcut_kullaniciyi_getir),
    db: Session = Depends(get_db),
):
    kitap = _kitabi_ve_sahiplik_kontrolunu_yap(kitap_id, mevcut_kullanici, db)
    guncellenecek_alanlar = veri.model_dump(exclude_unset=True)
    for alan, deger in guncellenecek_alanlar.items():
        setattr(kitap, alan, deger)
    db.commit()
    db.refresh(kitap)
    return kitap


@app.delete("/books/{kitap_id}", status_code=status.HTTP_204_NO_CONTENT, tags=["Kitaplar"])
def kitap_sil(
    kitap_id: int,
    mevcut_kullanici: KullaniciDB = Depends(mevcut_kullaniciyi_getir),
    db: Session = Depends(get_db),
):
    kitap = _kitabi_ve_sahiplik_kontrolunu_yap(kitap_id, mevcut_kullanici, db)
    db.delete(kitap)
    db.commit()
    return None


client = TestClient(app)
print(f"FastAPI uygulaması hazır. Toplam {len(app.openapi()['paths'])} endpoint tanımlı.")

FastAPI uygulaması hazır. Toplam 5 endpoint tanımlı.


## 6. Testler\n\n### Yardımcı Fonksiyonlar

In [6]:
def _benzersiz_kullanici_adi():
    return f"test_kullanici_{int(time.time() * 1_000_000) % 10_000_000}"


def _yeni_kullanici_ve_token(sifre: str = "guclu-sifre-123") -> str:
    kullanici_adi = _benzersiz_kullanici_adi()
    client.post("/register", json={"kullanici_adi": kullanici_adi, "sifre": sifre})
    giris_yaniti = client.post("/token", data={"username": kullanici_adi, "password": sifre})
    return giris_yaniti.json()["access_token"]

### Test 1 — Kayıt ve Giriş Akışı

In [7]:
kullanici_adi = _benzersiz_kullanici_adi()
kayit_yaniti = client.post("/register", json={"kullanici_adi": kullanici_adi, "sifre": "guclu-sifre-123"})
assert kayit_yaniti.status_code == 201, kayit_yaniti.text
assert kayit_yaniti.json()["kullanici_adi"] == kullanici_adi

# Aynı kullanıcı adıyla ikinci kayıt -> 400 beklenir.
tekrar_yaniti = client.post("/register", json={"kullanici_adi": kullanici_adi, "sifre": "baska-sifre"})
assert tekrar_yaniti.status_code == 400

giris_yaniti = client.post("/token", data={"username": kullanici_adi, "password": "guclu-sifre-123"})
assert giris_yaniti.status_code == 200
assert "access_token" in giris_yaniti.json()
print("[OK] Kayıt ve giriş akışı çalışıyor.")

[OK] Kayıt ve giriş akışı çalışıyor.


### Test 2 — Yanlış Şifre 401 Döner

In [8]:
kullanici_adi = _benzersiz_kullanici_adi()
client.post("/register", json={"kullanici_adi": kullanici_adi, "sifre": "dogru-sifre"})
yanit = client.post("/token", data={"username": kullanici_adi, "password": "yanlis-sifre"})
assert yanit.status_code == 401
print("[OK] Yanlış şifre 401 döndürüyor.")

[OK] Yanlış şifre 401 döndürüyor.


### Test 3 — Token Olmadan Kitap Eklenemez (401)

In [9]:
yanit = client.post("/books", json={"baslik": "Test Kitap", "yazar": "Test Yazar"})
assert yanit.status_code == 401
print("[OK] Kimliksiz istek reddediliyor.")

[OK] Kimliksiz istek reddediliyor.


### Test 4 — Uçtan Uca Kitap CRUD Akışı (Create → Read → Patch → Delete)

In [10]:
token = _yeni_kullanici_ve_token()
basliklar = {"Authorization": f"Bearer {token}"}

olustur_yaniti = client.post("/books", json={"baslik": "Suç ve Ceza", "yazar": "Dostoyevski", "yil": 1866}, headers=basliklar)
assert olustur_yaniti.status_code == 201
kitap = olustur_yaniti.json()
assert kitap["musait_mi"] is True
kitap_id = kitap["id"]
print(f"Oluşturulan kitap: {kitap}")

getir_yaniti = client.get(f"/books/{kitap_id}")
assert getir_yaniti.status_code == 200
assert getir_yaniti.json()["baslik"] == "Suç ve Ceza"

patch_yaniti = client.patch(f"/books/{kitap_id}", json={"musait_mi": False}, headers=basliklar)
assert patch_yaniti.status_code == 200
assert patch_yaniti.json()["musait_mi"] is False
assert patch_yaniti.json()["yazar"] == "Dostoyevski", "PATCH sadece gönderilen alanı değiştirmeli!"
print(f"PATCH sonrası: {patch_yaniti.json()}")

sil_yaniti = client.delete(f"/books/{kitap_id}", headers=basliklar)
assert sil_yaniti.status_code == 204

tekrar_getir_yaniti = client.get(f"/books/{kitap_id}")
assert tekrar_getir_yaniti.status_code == 404
print("[OK] Tam CRUD döngüsü (create/read/patch/delete) başarıyla çalıştı.")

Oluşturulan kitap: {'id': 1, 'baslik': 'Suç ve Ceza', 'yazar': 'Dostoyevski', 'yil': 1866, 'musait_mi': True, 'sahip_id': 3}
PATCH sonrası: {'id': 1, 'baslik': 'Suç ve Ceza', 'yazar': 'Dostoyevski', 'yil': 1866, 'musait_mi': False, 'sahip_id': 3}
[OK] Tam CRUD döngüsü (create/read/patch/delete) başarıyla çalıştı.


### Test 5 — Başkasının Kitabını Silemezsin (403 — BOLA Koruması)

In [11]:
token_a = _yeni_kullanici_ve_token()
token_b = _yeni_kullanici_ve_token()

olustur_yaniti = client.post(
    "/books", json={"baslik": "A'nın Kitabı", "yazar": "Biri"}, headers={"Authorization": f"Bearer {token_a}"}
)
kitap_id = olustur_yaniti.json()["id"]

sil_denemesi = client.delete(f"/books/{kitap_id}", headers={"Authorization": f"Bearer {token_b}"})
assert sil_denemesi.status_code == 403, "BOLA koruması çalışmıyor - başkasının kitabı silinebildi!"
print("[OK] BOLA koruması (bkz. 22.4.1, Bölüm 4) doğru çalışıyor: başkasının kitabı silinemiyor.")

[OK] BOLA koruması (bkz. 22.4.1, Bölüm 4) doğru çalışıyor: başkasının kitabı silinemiyor.


### Test 6 — Sayfalama ve Filtreleme

In [12]:
token = _yeni_kullanici_ve_token()
basliklar = {"Authorization": f"Bearer {token}"}
for i in range(15):
    client.post("/books", json={"baslik": f"Sayfalama Kitabı {i}", "yazar": "OrtakYazarX"}, headers=basliklar)

sayfa1 = client.get("/books", params={"sayfa": 1, "limit": 10, "yazar": "OrtakYazarX"})
assert sayfa1.status_code == 200
veri = sayfa1.json()
assert veri["toplam"] == 15
assert len(veri["kitaplar"]) == 10

sayfa2 = client.get("/books", params={"sayfa": 2, "limit": 10, "yazar": "OrtakYazarX"})
assert len(sayfa2.json()["kitaplar"]) == 5
print(f"[OK] Sayfalama doğru çalışıyor: toplam={veri['toplam']}, 1. sayfa={len(veri['kitaplar'])} kayıt, 2. sayfa={len(sayfa2.json()['kitaplar'])} kayıt.")

[OK] Sayfalama doğru çalışıyor: toplam=15, 1. sayfa=10 kayıt, 2. sayfa=5 kayıt.


### Test 7 — Geçersiz Veri 422 Döner (Otomatik Pydantic Doğrulaması)

In [13]:
token = _yeni_kullanici_ve_token()
basliklar = {"Authorization": f"Bearer {token}"}
# 'baslik' alanı ZORUNLU ama gönderilmedi.
yanit = client.post("/books", json={"yazar": "Biri"}, headers=basliklar)
assert yanit.status_code == 422
print("[OK] Eksik zorunlu alan otomatik olarak 422 ile reddedildi.")
print("\nTÜM TESTLER BAŞARIYLA TAMAMLANDI.")

[OK] Eksik zorunlu alan otomatik olarak 422 ile reddedildi.

TÜM TESTLER BAŞARIYLA TAMAMLANDI.


## 7. Bonus: Oluşan OpenAPI Şemasını İnceleme

FastAPI, hiçbir ek kod yazmadan, yukarıdaki tanımlardan OpenAPI şemasını otomatik üretir
(bkz. 22.7.1, Bölüm 1.2). Aşağıda, tanımlı tüm uç noktaları ve metodlarını programatik olarak
listeliyoruz — production'da bu şema `/docs` (Swagger UI) ve `/redoc` adreslerinde interaktif
olarak sunulur.

In [14]:
sema = app.openapi()
print(f"API Başlığı: {sema['info']['title']} (v{sema['info']['version']})")
print(f"Toplam endpoint sayısı: {len(sema['paths'])}\n")
for yol, metodlar in sema["paths"].items():
    for metod in metodlar:
        if metod in ("get", "post", "put", "patch", "delete"):
            print(f"  {metod.upper():6s} {yol}")

API Başlığı: Kütüphane Yönetim API'si (v1.0.0)
Toplam endpoint sayısı: 5

  POST   /register
  POST   /token
  GET    /me
  POST   /books
  GET    /books
  GET    /books/{kitap_id}
  PATCH  /books/{kitap_id}
  DELETE /books/{kitap_id}


## Sonuç

Katmanlı bir mimariyle (veri → şema → güvenlik → dependency → uç noktalar), gerçek bir veritabanına
yazan, JWT ile korunan, BOLA saldırılarına karşı korumalı, sayfalanan/filtrelenen, otomatik
doğrulama yapan tam bir REST API'yi sıfırdan inşa ettik ve 7 farklı senaryoyla test ettik.

**Bu API'yi gerçek bir sunucu olarak çalıştırmak için** (bu notebook'un dışında, bir terminalde):
```bash
uvicorn 22.6.2_kutuphane_api:app --reload
```
Sonra tarayıcıdan `http://127.0.0.1:8000/docs` adresini ziyaret edin (Swagger UI, bkz.
[22.7.1-API_Dokümantasyonu_ve_Test_Etme.pdf](../22.7-Dokümantasyon_ve_Test/22.7.1-API_Dokümantasyonu_ve_Test_Etme.pdf)).

Modülü tamamlamak için sırasıyla [22.7](../22.7-Dokümantasyon_ve_Test/), [22.8](../22.8-GraphQL_gRPC_ve_Webhooklar/),
[22.9](../22.9-API_Yönetimi_ve_Ölçekleme/) dosyalarını ve [22.10](../22.10-Genel_Tekrar_Soruları/)'daki
genel tekrar sorularını inceleyebilirsiniz.